In [1]:
!pip install dagshub mlflow neuralforecast -q

import pandas as pd
import numpy as np
import mlflow
import dagshub
from kaggle_secrets import UserSecretsClient
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST, NLinear
import warnings
warnings.filterwarnings('ignore')

# 1. DIRECT DAGSHUB INITIALIZATION
REPO_OWNER = "ejoba22"  
REPO_NAME = "walmart-sales-forecasting"

dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)

mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 87.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 88.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=8563636d-f576-465b-a206-b8c659b1f4dd&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=fc48deec4ba36e0b3f96234a3d4b198c4d85fac3c4e2993703221b632c0d7dcc




Accessing as tvada22

Initialized MLflow to track repo "ejoba22/walmart-sales-forecasting"

Repository ejoba22/walmart-sales-forecasting initialized!

In [2]:
# 2. DATA LOADING & TENSOR FORMATTING
DATA_BASE_PATH = '/kaggle/input/notebooks/elenejobava/walmart-eda-feature-engineering/'

train_fe = pd.read_parquet(DATA_BASE_PATH + 'train_features.parquet')

if 'Type' in train_fe.columns:
    train_fe = train_fe.drop(columns=['Type'])

# Format strictly for NeuralForecast
df_dl = train_fe.copy()
df_dl = df_dl.rename(columns={'Store_Dept': 'unique_id', 'Date': 'ds', 'Weekly_Sales': 'y'})
df_dl['weight'] = np.where(df_dl['IsHoliday'] == 1, 5, 1)

# PyTorch cannot compute gradients with NaNs. Fill cold starts with 0.
df_dl = df_dl.fillna(0)

df_dl = df_dl.sort_values(['unique_id', 'ds']).reset_index(drop=True)

# Time-Series Split (12 weeks)
split_date = df_dl['ds'].max() - pd.Timedelta(weeks=12)
train_df = df_dl[df_dl['ds'] <= split_date]
val_df = df_dl[df_dl['ds'] > split_date]

# Count how many weeks of history every single department has
series_lengths = train_df['unique_id'].value_counts()

# Keep only the IDs that have at least 52 weeks (our input_size requirement)
valid_ids = series_lengths[series_lengths >= 52].index

print(f"Dropping {len(series_lengths) - len(valid_ids)} departments that lack 52 weeks of history...")

# Filter the dataframes
train_df = train_df[train_df['unique_id'].isin(valid_ids)]
val_df = val_df[val_df['unique_id'].isin(valid_ids)]



Dropping 340 departments that lack 52 weeks of history...


In [3]:
# 3. METRICS
def calculate_wape(y_true, y_pred):
    return (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100

def calculate_wmae(y_true, y_pred, weights):
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [4]:
# 4. ARCHITECTURE SETUP & TRAINING

# 1. The Complex Transformer
model_patchtst = PatchTST(
    h=12, 
    input_size=52, 
    patch_len=4,          # Group 4 weeks (1 month) into a single patch
    stride=4,             # Move forward 4 weeks at a time
    max_steps=300,        # Keep steps low to prevent Kaggle timeout
    scaler_type='robust',
    random_seed=42
)

# 2. The Simple Linear Baseline
model_nlinear = NLinear(
    h=12, 
    input_size=52, 
    scaler_type='robust', # The normalization trick
    random_seed=42
)

# Nixtla can train both models in parallel/sequence automatically
nf = NeuralForecast(models=[model_nlinear, model_patchtst], freq='W-FRI')

nf.fit(df=train_df)

val_preds_df = nf.predict().reset_index()
results = val_df[['unique_id', 'ds', 'y', 'weight']].merge(val_preds_df, on=['unique_id', 'ds'], how='inner')

Seed set to 42
Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 2 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 1 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | tr

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=300` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [5]:
# --- Log NLinear ---
mlflow.set_experiment("NLinear_Training")
with mlflow.start_run(run_name="NLinear_Baseline"):
    wmae_nlin = calculate_wmae(results['y'], results['NLinear'], results['weight'])
    wape_nlin = calculate_wape(results['y'], results['NLinear'])
    
    mlflow.log_param("architecture", "NLinear")
    mlflow.log_param("input_size", 52)
    mlflow.log_metric("validation_WMAE", wmae_nlin)
    mlflow.log_metric("validation_WAPE_percent", wape_nlin)
    print(f"\n[NLinear] Validation WMAE: {wmae_nlin:.2f} | WAPE: {wape_nlin:.2f}%")


[NLinear] Validation WMAE: 1654.32 | WAPE: 10.08%
🏃 View run NLinear_Baseline at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/9/runs/3fc353778dab4883bf07d985c89af54c
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/9


In [6]:
# --- Log PatchTST ---
mlflow.set_experiment("PatchTST_Training")
with mlflow.start_run(run_name="PatchTST_Baseline"):
    wmae_patch = calculate_wmae(results['y'], results['PatchTST'], results['weight'])
    wape_patch = calculate_wape(results['y'], results['PatchTST'])
    
    mlflow.log_param("architecture", "PatchTST")
    mlflow.log_param("patch_len", 4)
    mlflow.log_metric("validation_WMAE", wmae_patch)
    mlflow.log_metric("validation_WAPE_percent", wape_patch)
    print(f"[PatchTST] Validation WMAE: {wmae_patch:.2f} | WAPE: {wape_patch:.2f}%")

[PatchTST] Validation WMAE: 1541.44 | WAPE: 9.46%
🏃 View run PatchTST_Baseline at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/10/runs/95f75ca87a9e4e7dad85e5fa0b3f98ef
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/10


In [8]:
# We will test 3 different training lengths
training_steps = [10, 300, 1000]
best_patch_wape = float('inf')

for steps in training_steps:
    print(f"\n--- Testing PatchTST with max_steps={steps} ---")
    
    with mlflow.start_run(run_name=f"PatchTST_Steps_{steps}"):
        
        # Initialize model with changing steps
        test_model = PatchTST(
            h=12, 
            input_size=52, 
            patch_len=4,
            stride=4,
            max_steps=steps,     
            scaler_type='robust',
            random_seed=42,
            accelerator='gpu',   # [THE FIX] Passed directly as a raw argument
            devices=1     
        )
        
        # Train
        nf_test = NeuralForecast(models=[test_model], freq='W-FRI')
        nf_test.fit(df=train_df)
        
        # Predict and merge
        test_preds = nf_test.predict().reset_index()
        test_results = val_df[['unique_id', 'ds', 'y', 'weight']].merge(test_preds, on=['unique_id', 'ds'], how='inner')
        
        # Calculate Validation WAPE & WMAE
        val_wape = calculate_wape(test_results['y'], test_results['PatchTST'])
        val_wmae = calculate_wmae(test_results['y'], test_results['PatchTST'], test_results['weight'])
        
        # Log to MLflow
        mlflow.log_param("architecture", "PatchTST")
        mlflow.log_param("max_steps", steps)
        mlflow.log_metric("validation_WMAE", val_wmae)
        mlflow.log_metric("validation_WAPE_percent", val_wape)
        
        print(f"Validation WMAE: {val_wmae:.2f} | WAPE: {val_wape:.2f}%")
        
        if val_wape < best_patch_wape:
            best_patch_wape = val_wape
            
print(f"\n✓ Best WAPE achieved: {best_patch_wape:.2f}%")



--- Testing PatchTST with max_steps=10 ---


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 421 K  | train
------------------------------------------------------------------
421 K     Trainable params
3         Non-trainable params
421 K     Total params
1.686     Total estimated model params size (MB)
93        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1000` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

Validation WMAE: 1474.82 | WAPE: 9.24%
🏃 View run PatchTST_Steps_1000 at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/0/runs/4d4f9559b844467dae7450240f00eaef
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/0

✓ Best WAPE achieved: 9.24%
